In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import cv2, random, os
from pathlib import Path
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import normalize
import math
import seaborn as sns
import numpy as np
import torchvision.transforms as T

In [3]:
train_df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/train.csv")
sub_df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/sample_submission.csv")

In [4]:
TRAIN_DIR = Path("/kaggle/input/competitions/landmark-recognition-2021/train")
TEST_DIR  = Path("/kaggle/input/competitions/landmark-recognition-2021/test")

In [5]:
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(42)
torch.backends.cudnn.benchmark = True

In [6]:
def make_subset(df, frac=0.05, seed=42):
    df = df.sample(frac=frac, random_state=seed).reset_index(drop=True)
    return df

def split_train_val_by_class(df, val_ratio=0.2, seed=42):
    train_parts = []
    val_parts = []

    rng = np.random.RandomState(seed)

    for cls, g in df.groupby("landmark_id"):
        g = g.sample(frac=1, random_state=seed).reset_index(drop=True)
        n = len(g)

        if n == 1:
            train_parts.append(g)
            continue

        n_val = max(1, int(round(n * val_ratio)))
        if n_val >= n:
            n_val = n - 1

        val_idx = rng.choice(n, size=n_val, replace=False)
        val_mask = np.zeros(n, dtype=bool)
        val_mask[val_idx] = True

        val_parts.append(g[val_mask])
        train_parts.append(g[~val_mask])

    train_df = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    val_df = pd.concat(val_parts).sample(frac=1, random_state=seed).reset_index(drop=True) if val_parts else pd.DataFrame(columns=df.columns)

    return train_df, val_df

In [7]:
DEBUG = True

CFG = {
    "backbone": "resnet18",
    "embedding_size": 256,
    "batch_size": 8,
    "image_size": 224,
    "epochs": 2,
    "lr": 1e-4,
    "margin": 0.3,
    "scale": 20.0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

In [8]:
if DEBUG:
    subset_frac = 0.05
    CFG["batch_size"] = 8
    CFG["image_size"] = 224
    CFG["epochs"] = 2
else:
    subset_frac = 1.0
    CFG["batch_size"] = 16
    CFG["image_size"] = 384
    CFG["epochs"] = 5

In [9]:
class LandmarkDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def load_image(self, img_id):
        path = self.img_dir / img_id[0] / img_id[1] / img_id[2] / f"{img_id}.jpg"
        img = Image.open(path).convert("RGB")
        return ImageOps.exif_transpose(img)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.load_image(row["id"])
        if self.transform:
            img = self.transform(img)
        label = int(row["label_idx"])
        return img, torch.tensor(label, dtype=torch.long)

In [10]:
train_tfms = T.Compose([
    T.Resize((CFG["image_size"], CFG["image_size"])),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.RandomHorizontalFlip()
])

val_tfms = T.Compose([
    T.Resize((CFG["image_size"], CFG["image_size"])),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomRotation(degrees=15),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    T.RandomHorizontalFlip()
])

In [11]:
unique_labels = train_df["landmark_id"].unique()
label2idx = {l: i for i, l in enumerate(unique_labels)}

train_df["label_idx"] = train_df["landmark_id"].map(label2idx).astype(int)

In [12]:
train_ds = LandmarkDataset(train_df, TRAIN_DIR, transform=train_tfms)

pin_memory = torch.cuda.is_available()
train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

In [13]:
class GeM(nn.Module):
    def __init__(self, p=3.0, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * float(p))
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=(-1, -2)).pow(1.0 / self.p)

In [14]:
class EmbeddingNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            CFG["backbone"],
            pretrained=True,
            num_classes=0
        )
        self.pool = GeM(3.0)
        self.fc = nn.Linear(self.backbone.num_features, CFG["embedding_size"])
        self.bn = nn.BatchNorm1d(CFG["embedding_size"])
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        x = self.backbone.forward_features(x)
        x = self.pool(x)
        x = self.fc(x)
        x = self.dropout(x)
        x = self.bn(x)
        return F.normalize(x, dim=1)

In [38]:
class ArcFaceLoss(nn.Module):
    def __init__(self, num_classes, embedding_size, margin=0.5, scale=64.0):
        super().__init__()
        self.num_classes = num_classes
        self.margin = margin
        self.scale = scale
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        weight_norm = F.normalize(self.weight, dim=1)
        embeddings_norm = F.normalize(embeddings, dim=1)
        cosine = F.linear(embeddings_norm, weight_norm)

        labels = labels.long()
        assert labels.min() >= 0 and labels.max() < self.num_classes, "Метка вне допустимого диапазона"

        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, labels.view(-1, 1), 1)
        theta_with_margin = theta + one_hot * self.margin
        output = torch.cos(theta_with_margin) * self.scale
        return F.cross_entropy(output, labels)

In [74]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class PartialArcFaceLoss(nn.Module):
    def __init__(self, num_classes, embedding_size, margin=0.3, scale=30.0,
                 neg_class_num=2048):
        super().__init__()
        self.num_classes = num_classes
        self.margin = margin
        self.scale = scale
        self.neg_class_num = neg_class_num

        self.W = nn.Parameter(torch.FloatTensor(num_classes, embedding_size))
        nn.init.xavier_uniform_(self.W)

        self.cos_m = math.cos(margin)
        self.sin_m = math.sin(margin)
        self.th = math.cos(math.pi - margin)
        self.mm = math.sin(math.pi - margin) * margin

    def forward(self, embeddings, labels):
        W_norm = F.normalize(self.W)

        pos_classes = torch.unique(labels)
        if self.neg_class_num > 0 and len(pos_classes) < self.neg_class_num:
            all_classes = torch.arange(self.num_classes, device=embeddings.device)
            mask = torch.ones(self.num_classes, dtype=torch.bool, device=embeddings.device)
            mask[pos_classes] = False
            neg_pool = all_classes[mask]
            if len(neg_pool) > 0:
                num_neg = min(self.neg_class_num - len(pos_classes), len(neg_pool))
                neg_classes = neg_pool[torch.randperm(len(neg_pool))[:num_neg]]
            else:
                neg_classes = torch.tensor([], dtype=torch.long, device=embeddings.device)
            sub_classes = torch.cat([pos_classes, neg_classes])
        else:
            sub_classes = torch.arange(self.num_classes, device=embeddings.device)

        sub_W = W_norm[sub_classes]

        sub_class_to_idx = {cls.item(): i for i, cls in enumerate(sub_classes)}
        mapped_labels = torch.tensor([sub_class_to_idx[l.item()] for l in labels],
                                     device=labels.device, dtype=torch.long)

        cosine = F.linear(embeddings, sub_W)

        sine = torch.sqrt((1.0 - cosine**2).clamp(0, 1))
        phi = cosine * self.cos_m - sine * self.sin_m
        phi = torch.where(cosine > self.th, phi, cosine - self.mm)

        one_hot = torch.zeros_like(cosine)
        one_hot.scatter_(1, mapped_labels.view(-1, 1), 1)

        logits = one_hot * phi + (1 - one_hot) * cosine
        logits *= self.scale

        loss = F.cross_entropy(logits, mapped_labels)
        return loss

In [ ]:
full_df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/train.csv")
subset_df = make_subset(full_df, frac=1.0, seed=42)
train_df, val_df = split_train_val_by_class(subset_df, val_ratio=0.2, seed=42)

unique_labels = sorted(train_df["landmark_id"].unique())

label2idx = {l: i for i, l in enumerate(unique_labels)}
idx2label = {i: l for l, i in label2idx.items()}

num_classes = len(label2idx)
print(f"Total classes: {num_classes}")

train_df["label_idx"] = train_df["landmark_id"].map(label2idx)

val_df = val_df[val_df["landmark_id"].isin(label2idx)].copy()
val_df["label_idx"] = val_df["landmark_id"].map(label2idx)

print("Train min/max:", train_df["label_idx"].min(), train_df["label_idx"].max())
print("Val min/max:", val_df["label_idx"].min(), val_df["label_idx"].max())
print("Num classes:", len(label2idx))

Total classes: 81313
Train min/max: 0 81312
Val min/max: 0 81312
Num classes: 81313


In [17]:
train_ds = LandmarkDataset(train_df, TRAIN_DIR, transform=train_tfms)
val_ds   = LandmarkDataset(val_df, TRAIN_DIR, transform=val_tfms)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=pin_memory,
    drop_last=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=2,
    pin_memory=pin_memory,
)

In [18]:
def average_precision(gt_label, preds):
    for i, (pred_label, _) in enumerate(preds, 1):
        if pred_label == gt_label:
            return 1.0 / i
    return 0.0

In [19]:
def compute_gap(prediction_strings, gt_labels):
    ap_sum = 0.0

    for pred_str, gt in zip(prediction_strings, gt_labels):
        if not pred_str.strip():
            ap = 0.0
        else:
            parts = pred_str.strip().split()
            pred_list = []

            for p in range(0, len(parts), 2):
                lbl = int(parts[p])
                conf = float(parts[p+1])
                pred_list.append((lbl, conf))

            pred_list.sort(key=lambda x: x[1], reverse=True)
            ap = average_precision(gt, pred_list)

        ap_sum += ap

    return ap_sum / len(gt_labels)

In [20]:
def get_prediction_strings(logits, topk=5):
    probs = logits.softmax(dim=1)
    topk_vals, topk_idx = probs.topk(topk, dim=1)

    results = []

    for vals, idxs in zip(topk_vals, topk_idx):
        parts = []
        for v, i in zip(vals, idxs):
            parts.append(f"{int(i)} {float(v)}")
        results.append(" ".join(parts))

    return results

In [ ]:
@torch.no_grad()
def get_embeddings(model, loader):
    model.eval()

    all_embs = []
    all_labels = []

    for imgs, labels in loader:
        imgs = imgs.to(CFG["device"])

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            emb = model(imgs)

        all_embs.append(emb.cpu())
        all_labels.extend(labels.numpy())

    all_embs = torch.cat(all_embs, dim=0)  # [N, D]
    all_labels = torch.tensor(all_labels)

    return all_embs, all_labels

In [22]:
@torch.no_grad()
def evaluate_gap_knn(model, train_loader, val_loader, topk=5):
    model.eval()

    train_embs, train_labels = get_embeddings(model, train_loader)
    train_embs = F.normalize(train_embs, dim=1)
    train_labels_np = train_labels.cpu().numpy()

    all_preds = []
    all_gts = []

    for imgs, labels in val_loader:
        imgs = imgs.to(CFG["device"])
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            val_embs = model(imgs).cpu()
        val_embs = F.normalize(val_embs, dim=1)

        sim = val_embs @ train_embs.T

        topk_vals, topk_idx = sim.topk(topk, dim=1)

        for vals, idxs, gt in zip(topk_vals, topk_idx, labels):
            unique = {}
            for v, idx in zip(vals, idxs):
                lbl = int(train_labels_np[idx])
                conf = float(v)
                if lbl not in unique or conf > unique[lbl]:
                    unique[lbl] = conf

            sorted_items = sorted(unique.items(), key=lambda x: x[1], reverse=True)[:topk]
            parts = [f"{lbl} {conf}" for lbl, conf in sorted_items]
            all_preds.append(" ".join(parts))
            all_gts.append(gt.item())

    for i in range(min(5, len(all_preds))):
        print(f"\n--- Sample {i} (GT={all_gts[i]}) ---")
        parts = all_preds[i].split()
        for p in range(0, len(parts), 2):
            print(f"  Pred: {parts[p]} conf={float(parts[p+1]):.4f}")

    return compute_gap(all_preds, all_gts)

In [23]:
CFG["image_size"] = 256
CFG["batch_size"] = 16

train_tfms = T.Compose([
    T.Resize((128, 128)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_loader = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True
)

In [42]:
CFG["epochs"] = 10

In [25]:
CFG["backbone"] = "tf_efficientnet_b3"

In [27]:
print(CFG)

{'backbone': 'tf_efficientnet_b3', 'embedding_size': 256, 'batch_size': 16, 'image_size': 256, 'epochs': 5, 'lr': 0.0001, 'margin': 0.3, 'scale': 20.0, 'device': 'cuda'}


In [28]:
import time

start = time.time()
imgs, labels = next(iter(train_loader))
print("Batch load time:", time.time() - start)

Batch load time: 0.3380091190338135


In [ ]:
for batch in train_loader:
    images, labels = batch
    assert labels.min() >= 0 and labels.max() < len(label2idx), \
        f"Ошибка! Метки от {labels.min()} до {labels.max()}, а классов всего {len(label2idx)}"

In [30]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [36]:
import sys
from tqdm import tqdm

In [ ]:
model = EmbeddingNet().to(CFG["device"])
criterion = PartialArcFaceLoss(
    num_classes=len(label2idx),
    embedding_size=CFG["embedding_size"],
    margin=CFG["margin"],
    scale=CFG["scale"],
    neg_class_num=2048
).to(CFG["device"])

optimizer = torch.optim.AdamW(
    list(model.parameters()) + list(criterion.parameters()),
    lr=CFG["lr"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

imgs, labels = next(iter(train_loader))
imgs = imgs.to(CFG["device"])
labels = labels.to(CFG["device"])

with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
    emb = model(imgs)
    loss = criterion(emb, labels)

if torch.isnan(loss):
    print("NaN detected!")
    raise ValueError("NaN in loss")

print("Sanity check OK:", emb.shape, loss.item())

best_val_gap = 0.0

for epoch in range(CFG["epochs"]):
    model.train()
    criterion.train()
    total_loss = 0.0
    total = 0

    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}', file=sys.stdout, mininterval=1.0)
    for imgs, labels in loop:
        imgs = imgs.to(CFG["device"], non_blocking=True)
        labels = labels.to(CFG["device"], non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            embeddings = model(imgs)
            loss = criterion(embeddings, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = labels.size(0)
        total_loss += loss.item() * bs
        total += bs

        if total % (10*bs) == 0:
            loop.set_postfix(loss=loss.item())
            sys.stdout.flush()

    train_loss = total_loss / max(total, 1)


    val_gap = evaluate_gap_knn(model, train_loader, val_loader, topk=5)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_gap={val_gap:.4f}")

    if val_gap > best_val_gap:
        best_val_gap = val_gap
        torch.save({
            "model": model.state_dict(),
            "criterion": criterion.state_dict(),
            "label2idx": label2idx,
            "idx2label": idx2label,
            "cfg": CFG,
        }, "/kaggle/working/best_arcface_debug.pth")
        print("Saved best checkpoint")

    scheduler.step()

Sanity check OK: torch.Size([16, 256]) 14.389603614807129
Epoch 1: 100%|██████████| 450/450 [02:58<00:00,  2.52it/s, loss=10.1]

--- Sample 0 (GT=344) ---
  Pred: 409 conf=0.9271
  Pred: 237 conf=0.9054
  Pred: 294 conf=0.9053
  Pred: 494 conf=0.9038
  Pred: 141 conf=0.9037

--- Sample 1 (GT=115) ---
  Pred: 457 conf=0.9010
  Pred: 127 conf=0.8982
  Pred: 294 conf=0.8954
  Pred: 441 conf=0.8915
  Pred: 30 conf=0.8866

--- Sample 2 (GT=80) ---
  Pred: 431 conf=0.9458
  Pred: 207 conf=0.9446
  Pred: 139 conf=0.9421
  Pred: 316 conf=0.9318
  Pred: 360 conf=0.9293

--- Sample 3 (GT=486) ---
  Pred: 102 conf=0.7907
  Pred: 310 conf=0.7838
  Pred: 66 conf=0.7800
  Pred: 159 conf=0.7747
  Pred: 127 conf=0.7725

--- Sample 4 (GT=431) ---
  Pred: 55 conf=0.9057
  Pred: 401 conf=0.8874
  Pred: 200 conf=0.8858
  Pred: 385 conf=0.8810
Epoch 1: train_loss=12.4053 val_gap=0.1559
Saved best checkpoint
Epoch 2: 100%|██████████| 450/450 [02:45<00:00,  2.73it/s, loss=5.93]

--- Sample 0 (GT=344) ---
  P

In [44]:
index_tfms = T.Compose([
    T.Resize((CFG["image_size"], CFG["image_size"])),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_index_ds = LandmarkDataset(train_df, TRAIN_DIR, transform=index_tfms)
train_index_loader = DataLoader(
    train_index_ds,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [57]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [58]:
from tqdm import tqdm
import faiss

In [ ]:
device = CFG["device"]
@torch.no_grad()
def get_all_embeddings(loader, model):
    model.eval()
    all_embs, all_labels = [], []
    for imgs, labels in tqdm(loader, desc="Extracting train embeddings"):
        imgs = imgs.to(device)
        emb = model(imgs)
        all_embs.append(emb.cpu())
        all_labels.extend(labels.numpy())
    return torch.cat(all_embs, dim=0), np.array(all_labels)

train_embs, train_labels_idx = get_all_embeddings(train_index_loader, model)
train_embs = F.normalize(train_embs, dim=1).numpy().astype('float32')
print(f"Train embeddings shape: {train_embs.shape}")

Extracting train embeddings: 100%|██████████| 1578/1578 [06:36<00:00,  3.98it/s]


Train embeddings shape: (100946, 256)


In [80]:
dim = train_embs.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(train_embs)
print(f"Index built with {index.ntotal} vectors")

Index built with 100946 vectors


In [81]:
class TestDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def load_image(self, img_id):
        path = self.img_dir / img_id[0] / img_id[1] / img_id[2] / f"{img_id}.jpg"
        img = Image.open(path).convert("RGB")
        return ImageOps.exif_transpose(img)

    def __getitem__(self, idx):
        img_id = self.df.loc[idx, "id"]
        img = self.load_image(img_id)
        if self.transform:
            img = self.transform(img)
        return img, img_id

test_tfms = T.Compose([
    T.Resize((CFG["image_size"], CFG["image_size"])),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_df = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/sample_submission.csv")
test_ds = TestDataset(test_df, TEST_DIR, transform=test_tfms)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [ ]:
from torchvision.transforms import functional as TF

@torch.no_grad()
def predict_test_tta(model, test_loader, faiss_index, train_labels_idx, idx2label, topk=100, threshold=0.0):
    model.eval()
    test_ids = []
    pred_strings = []

    for imgs, img_ids in tqdm(test_loader, desc="Predicting test with TTA"):
        imgs = imgs.to(device)  # [B, C, H, W]

        imgs_tta = []
        imgs_tta.append(imgs)
        imgs_tta.append(torch.flip(imgs, dims=[3]))
        imgs_tta.append(TF.rotate(imgs, angle=15, fill=0))
        imgs_tta.append(TF.rotate(imgs, angle=-15, fill=0))

        # (B*4, C, H, W)
        imgs_stacked = torch.cat(imgs_tta, dim=0)

        emb_all = model(imgs_stacked)                # [B*4, D]
        emb_all = F.normalize(emb_all, dim=1)

        B = imgs.size(0)
        emb_avg = emb_all.view(4, B, -1).mean(dim=0) # [B, D]

        emb_np = emb_avg.cpu().numpy().astype('float32')
        D, I = faiss_index.search(emb_np, topk)

        for i in range(B):
            label_conf = {}
            for j in range(topk):
                lbl_idx = train_labels_idx[I[i, j]]
                lbl = idx2label[lbl_idx]
                conf = float(D[i, j])
                if lbl not in label_conf or conf > label_conf[lbl]:
                    label_conf[lbl] = conf
            sorted_items = sorted(label_conf.items(), key=lambda x: x[1], reverse=True)
            if sorted_items and sorted_items[0][1] >= threshold:
                pred_str = " ".join(f"{lbl} {conf:.6f}" for lbl, conf in sorted_items[:100])
            else:
                pred_str = ""
            pred_strings.append(pred_str)
            test_ids.append(img_ids[i])

    return test_ids, pred_strings
test_ids, pred_strings = predict_test_tta(model, test_loader, index, train_labels_idx, idx2label,
                                      topk=100, threshold=0.3)

Predicting test with TTA: 100%|██████████| 162/162 [02:48<00:00,  1.04s/it]


In [76]:
@torch.no_grad()
def predict_test(model, test_loader, faiss_index, train_labels_idx, idx2label, topk=100, threshold=0.0):
    model.eval()
    test_ids = []
    pred_strings = []

    for imgs, img_ids in tqdm(test_loader, desc="Predicting test"):
        imgs = imgs.to(device)
        emb1 = model(imgs)
        emb2 = model(torch.flip(imgs, dims=[3]))
        emb = (emb1 + emb2) / 2.0
        emb = F.normalize(emb, dim=1).cpu().numpy().astype('float32')

        D, I = faiss_index.search(emb, topk)

        for i in range(len(D)):
            label_conf = {}
            for j in range(topk):
                lbl_idx = train_labels_idx[I[i, j]]
                lbl = idx2label[lbl_idx]
                conf = float(D[i, j])
                if lbl not in label_conf or conf > label_conf[lbl]:
                    label_conf[lbl] = conf
            sorted_items = sorted(label_conf.items(), key=lambda x: x[1], reverse=True)
            if sorted_items and sorted_items[0][1] >= threshold:
                pred_str = " ".join(f"{lbl} {conf:.6f}" for lbl, conf in sorted_items[:100])
            else:
                pred_str = ""
            pred_strings.append(pred_str)
            test_ids.append(img_ids[i])

    return test_ids, pred_strings

test_ids, pred_strings = predict_test(model, test_loader, index, train_labels_idx, idx2label,
                                      topk=100, threshold=0.3)

Predicting test: 100%|██████████| 162/162 [01:27<00:00,  1.85it/s]


In [83]:
submission = pd.DataFrame({"id": test_ids, "landmarks": pred_strings})
sample_sub = pd.read_csv("/kaggle/input/competitions/landmark-recognition-2021/sample_submission.csv")
submission = sample_sub[["id"]].merge(submission, on="id", how="left")
submission["landmarks"] = submission["landmarks"].fillna("")
submission.to_csv("submission_minimal.csv", index=False)
print("Файл submission_minimal.csv сохранён!")

Файл submission_minimal.csv сохранён!


In [84]:
submission.head()

,id,landmarks
0,00084cdf8f600d00,645 0.324602 1012 0.323186 455 0.321036 1739 0...
1,000b15b043eb8cf0,115 0.393511 337 0.373225 2054 0.361808 2361 0...
2,0011a52f9b948fd2,
3,00141b8a5a729084,1546 0.434354 2053 0.433113 2208 0.424893 1049...
4,0018aa4b92532b77,2220 0.329109 956 0.325787 1428 0.320757 2185 ...


In [85]:
import pandas as pd

info_path = "/kaggle/input/datasets/lalala555/category/train_label_to_category.csv"

info_df = pd.read_csv(info_path)

id_to_name = dict(zip(info_df["landmark_id"], info_df["category"]))

print(f"Загружено {len(id_to_name)} категорий")

Загружено 203094 категорий


In [86]:
import pandas as pd
from urllib.parse import unquote

info_path = "/kaggle/input/datasets/lalala555/category/train_label_to_category.csv"
info_df = pd.read_csv(info_path)

def clean_category_name(category_url: str) -> str:
    """
    Из URL вида:
    http://commons.wikimedia.org/wiki/Category:Walfriduskerk
    получаем строку:
    Walfriduskerk
    """
    name_part = category_url.rsplit('Category:', 1)[-1]
    name_part = name_part.rstrip('/')
    name_part = unquote(name_part)
    name_part = name_part.replace('_', ' ')
    return name_part

id_to_name = {}
for _, row in info_df.iterrows():
    lid = row['landmark_id']
    raw = row['category']
    id_to_name[lid] = clean_category_name(raw)

print(f"Загружено {len(id_to_name)} читаемых названий")

Загружено 203094 читаемых названий


In [87]:
for idx in range(5):
    img_id = test_ids[idx]
    preds = pred_strings[idx]

    print(f"\n===== Изображение {img_id} =====")
    if not preds.strip():
        print("  Нет предсказаний")
        continue

    parts = preds.strip().split()
    for p in range(0, len(parts), 2):
        landmark_id_str = parts[p]
        conf_str = parts[p+1]
        landmark_id = int(landmark_id_str)
        name = id_to_name.get(landmark_id, f"Неизвестный ID {landmark_id}")
        print(f"  {name:<40s}  уверенность: {float(conf_str):.4f}")


===== Изображение 00084cdf8f600d00 =====
  Arboretum in Alcsút                       уверенность: 0.3246
  Markgräfliches Opernhaus                  уверенность: 0.3232
  Holy Trinity Church in Ostankino (Moscow)  уверенность: 0.3210
  Little Belt Bridge (1970)                 уверенность: 0.3204
  HMS M33 (ship, 1915)                      уверенность: 0.3191
  Teleki Library                            уверенность: 0.3171
  Sant Joan de Llorac                       уверенность: 0.3171
  Cape Egmont Lighthouse                    уверенность: 0.3162
  Valle de la Luna (Chile)                  уверенность: 0.3156
  Museum of the Gorge, Ironbridge           уверенность: 0.3130
  Planetarium Bad Salzungen                 уверенность: 0.3127
  Drachenfels (Siebengebirge)               уверенность: 0.3100
  Ossiacher See                             уверенность: 0.3071
  Haus Auensee                              уверенность: 0.3058
  Bundesdenkmal des Bundes Deutscher Radfahrer  уверенность: 